# Exercise 3.1: Ingestion Patterns and Concurrency

In this exercise, you'll learn about different write patterns in Apache Iceberg using the NYC Yellow Taxi dataset:
- **Batch vs Streaming**: Understand different ingestion patterns
- **Copy-on-Write vs Merge-on-Read**: Compare performance tradeoffs
- **Row-Level Operations**: Updates, deletes, and merges
- **Concurrency**: Handle concurrent writes and conflicts

## Learning Objectives
- Implement batch and streaming-style writes
- Compare COW and MOR strategies
- Use MERGE operations for upserts
- Understand and resolve write conflicts

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time
from datetime import datetime, timedelta
import random

spark = SparkSession.builder \
    .appName("IngestionPatterns") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.ingestion")
print("Namespace 'ingestion' created!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June through August 2023**.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6, 7, 8]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO -- skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~45MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nAll taxi data ready in MinIO!")

## Part 1: Batch vs Streaming Ingestion Patterns

### Batch Ingestion Pattern

Large infrequent writes. Typical for daily ETL jobs.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_trips")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_trips
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet` WHERE 1=0
""")

print("Taxi trips table created!")

In [ ]:
june_df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet")

print(f"Batch loading {june_df.count():,} June trips...")

start = time.time()
june_df.writeTo("polaris.ingestion.taxi_trips").append()
batch_time = time.time() - start

print(f"Batch write completed in {batch_time:.2f} seconds")

In [ ]:
print("Snapshots after batch write:")
spark.sql("""
    SELECT committed_at, operation, summary['added-records'] as records_added
    FROM polaris.ingestion.taxi_trips.snapshots
    ORDER BY committed_at
""").show(truncate=False)

### Streaming Helper

> **Note:** You do not need to understand the implementation of this helper. It uses Spark Structured Streaming to read data from a staging directory and stream it into an Iceberg table in micro-batches. Adjust the parameters to control the streaming behavior.

In [ ]:
import shutil, os

def run_streaming_ingest(source_path, table_name,
                         num_source_files=10, max_files_per_trigger=2):
    """Stream parquet data into an Iceberg table using Spark Structured Streaming.

    Parameters:
        source_path:           S3 path to a source parquet file
        table_name:            Target Iceberg table (must already exist)
        num_source_files:      Split source into this many small files for streaming
        max_files_per_trigger: Files read per micro-batch (fewer = more snapshots)

    Returns the number of new snapshots created.
    """
    staging = "s3a://warehouse/staging/streaming_input"
    checkpoint = "/tmp/iceberg_streaming_checkpoint"

    if os.path.exists(checkpoint):
        shutil.rmtree(checkpoint)

    df = spark.read.parquet(source_path)
    schema = df.schema
    total_rows = df.count()

    print(f"Staging {total_rows:,} rows as {num_source_files} files...")
    df.repartition(num_source_files).write.mode("overwrite").parquet(staging)

    before = spark.sql(
        f"SELECT COUNT(*) FROM {table_name}.snapshots"
    ).collect()[0][0]

    print(f"Streaming with maxFilesPerTrigger={max_files_per_trigger} "
          f"(expect ~{num_source_files // max_files_per_trigger} micro-batches)...")
    start = time.time()

    query = (spark.readStream
             .schema(schema)
             .option("maxFilesPerTrigger", max_files_per_trigger)
             .parquet(staging)
             .writeStream
             .format("iceberg")
             .outputMode("append")
             .option("checkpointLocation", checkpoint)
             .toTable(table_name))

    query.processAllAvailable()
    query.stop()

    elapsed = time.time() - start
    after = spark.sql(
        f"SELECT COUNT(*) FROM {table_name}.snapshots"
    ).collect()[0][0]
    new_snapshots = after - before

    print(f"Done in {elapsed:.2f}s — created {new_snapshots} new snapshots")
    return new_snapshots

print("Streaming helper loaded!")

### Streaming Ingestion Pattern

Small frequent writes. Typical for real-time streaming.

In [ ]:
streaming_snapshots = run_streaming_ingest(
    source_path="s3a://warehouse/raw/yellow_tripdata_2023-07.parquet",
    table_name="polaris.ingestion.taxi_trips",
    num_source_files=10,
    max_files_per_trigger=2
)

In [ ]:
snapshot_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.ingestion.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Total snapshots: {snapshot_count}")

Streaming creates many small snapshots compared to batch's single large snapshot.

### Try It: Adjust Streaming Parameters

Use `run_streaming_ingest` to stream August data with different settings. Try changing `num_source_files` (how many files the source is split into) and `max_files_per_trigger` (how many files are read per micro-batch). More micro-batches means more snapshots and more small files — the classic streaming trade-off.

In [ ]:
# Try with many small micro-batches (more snapshots, more files):
# run_streaming_ingest(
#     source_path="s3a://warehouse/raw/yellow_tripdata_2023-08.parquet",
#     table_name="polaris.ingestion.taxi_trips",
#     num_source_files=20,
#     max_files_per_trigger=1
# )

# Or fewer, larger micro-batches (fewer snapshots):
# run_streaming_ingest(
#     source_path="s3a://warehouse/raw/yellow_tripdata_2023-08.parquet",
#     table_name="polaris.ingestion.taxi_trips",
#     num_source_files=20,
#     max_files_per_trigger=10
# )

# Compare the results
# spark.sql("SELECT COUNT(*) as snapshots FROM polaris.ingestion.taxi_trips.snapshots").show()
# spark.sql("SELECT COUNT(*) as files FROM polaris.ingestion.taxi_trips.files").show()

## Part 2: Copy-on-Write vs Merge-on-Read

Unlike traditional databases (MySQL, Postgres) that modify rows in place, Iceberg stores data in immutable files on object storage. To "update" a row, it must either rewrite the entire file containing that row, or log the change separately. This is the fundamental tradeoff of file-based table formats, and Iceberg gives you two strategies:

**COW (Copy-on-Write)** rewrites entire data files on updates. Slower writes, faster reads.
**MOR (Merge-on-Read)** writes small delete files. Faster writes, slower reads (merge overhead at read time).

> **Note on benchmarks:** The timings in this section are meant to illustrate the *relative* tradeoffs between COW and MOR. Absolute numbers will vary based on your local environment (CPU, memory, disk speed, Docker resource allocation). In production with distributed clusters and larger datasets, the differences are typically more pronounced.

### Create COW and MOR Tables

In [ ]:
for mode_suffix, mode_value in [('cow', 'copy-on-write'), ('mor', 'merge-on-read')]:
    spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.taxi_{mode_suffix}")
    spark.sql(f"""
        CREATE TABLE polaris.ingestion.taxi_{mode_suffix}
        USING iceberg
        TBLPROPERTIES (
            'write.update.mode' = '{mode_value}',
            'write.delete.mode' = '{mode_value}',
            'write.merge.mode' = '{mode_value}'
        )
        AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
    """)

row_count = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_cow").collect()[0][0]
size_mb = spark.sql("""
    SELECT ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 1)
    FROM polaris.ingestion.taxi_cow.files
""").collect()[0][0]
print(f"COW and MOR tables created with {row_count:,} rows each (~{size_mb} MB)")

### Compare UPDATE Performance

In [ ]:
start = time.time()
spark.sql("""
    UPDATE polaris.ingestion.taxi_cow
    SET fare_amount = fare_amount * 1.10
    WHERE trip_distance > 10
""")
cow_time = time.time() - start
print(f"COW UPDATE took {cow_time:.3f} seconds")

In [ ]:
start = time.time()
spark.sql("""
    UPDATE polaris.ingestion.taxi_mor
    SET fare_amount = fare_amount * 1.10
    WHERE trip_distance > 10
""")
mor_time = time.time() - start
print(f"MOR UPDATE took {mor_time:.3f} seconds")

if mor_time < cow_time:
    print(f"\nMOR was {cow_time/mor_time:.1f}x faster for writes")
else:
    print(f"\nCOW was {mor_time/cow_time:.1f}x faster for writes (small table effect)")

### Examine File Structure

In [ ]:
print("COW data files:")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           file_size_in_bytes, record_count
    FROM polaris.ingestion.taxi_cow.files
""").show(truncate=False)

In [ ]:
print("MOR data files:")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           file_size_in_bytes, record_count
    FROM polaris.ingestion.taxi_mor.files
""").show(truncate=False)

print("\nMOR delete files:")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           file_size_in_bytes, record_count
    FROM polaris.ingestion.taxi_mor.delete_files
""").show(truncate=False)

**COW** has only a **single data file**. The update rewrote the entire file, producing one new file that contains both the modified rows (with the 10% fare increase) and the unchanged rows. The original file was replaced.

**MOR** kept the original data file and wrote a small **delete file** alongside it. The delete file marks which rows were changed, and a new data file contains only the updated rows. At read time Spark merges them together.

### Compare READ Performance

In [ ]:
start = time.time()
spark.sql("SELECT * FROM polaris.ingestion.taxi_cow").collect()
cow_read = time.time() - start

start = time.time()
spark.sql("SELECT * FROM polaris.ingestion.taxi_mor").collect()
mor_read = time.time() - start

print(f"COW READ: {cow_read:.3f}s")
print(f"MOR READ: {mor_read:.3f}s")
if cow_read < mor_read:
    print(f"\nCOW reads were {mor_read/cow_read:.1f}x faster (no merge overhead)")
else:
    print(f"\nResults similar at this scale")

### Try It: How Does Selectivity Affect COW vs MOR?

The update above touched rows where `trip_distance > 10` — a small fraction of the data. What happens when you update a larger or smaller portion?

Try changing `update_pct` below and re-running. Some examples to try:

| `update_pct` | Approximate Filter | Rows Affected |
|---|---|---|
| `1` | Longest 1% of trips | ~33K rows |
| `10` | Longer-than-average trips | ~330K rows |
| `50` | Half the table | ~1.6M rows |
| `90` | Nearly everything | ~3M rows |

**Prediction:** MOR should shine at low percentages (small delete files). As the percentage grows, COW's single-file rewrite becomes competitive because MOR accumulates large delete files that slow reads.

In [ ]:
# update_pct = 10  # <-- CHANGE THIS: 1, 10, 50, 90
#
# # Recreate fresh COW and MOR tables so each run starts clean
# for mode_suffix, mode_value in [('cow', 'copy-on-write'), ('mor', 'merge-on-read')]:
#     spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.taxi_{mode_suffix}")
#     spark.sql(f"""
#         CREATE TABLE polaris.ingestion.taxi_{mode_suffix}
#         USING iceberg
#         TBLPROPERTIES (
#             'write.update.mode' = '{mode_value}',
#             'write.delete.mode' = '{mode_value}',
#             'write.merge.mode' = '{mode_value}'
#         )
#         AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
#     """)
#
# total = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_cow").collect()[0][0]
#
# # Use PERCENT_RANK to select exactly the desired fraction of rows
# threshold = spark.sql(f"""
#     SELECT PERCENTILE_APPROX(trip_distance, {(100 - update_pct) / 100.0})
#     FROM polaris.ingestion.taxi_cow
# """).collect()[0][0]
#
# affected = spark.sql(f"""
#     SELECT COUNT(*) FROM polaris.ingestion.taxi_cow WHERE trip_distance >= {threshold}
# """).collect()[0][0]
# print(f"Updating rows with trip_distance >= {threshold:.2f}")
# print(f"Affects {affected:,} / {total:,} rows ({100*affected/total:.1f}%)")
# print("=" * 60)
#
# # --- Write benchmark ---
# start = time.time()
# spark.sql(f"UPDATE polaris.ingestion.taxi_cow SET fare_amount = fare_amount * 1.10 WHERE trip_distance >= {threshold}")
# cow_write = time.time() - start
#
# start = time.time()
# spark.sql(f"UPDATE polaris.ingestion.taxi_mor SET fare_amount = fare_amount * 1.10 WHERE trip_distance >= {threshold}")
# mor_write = time.time() - start
#
# # --- Read benchmark ---
# start = time.time()
# spark.sql("SELECT * FROM polaris.ingestion.taxi_cow").collect()
# cow_read = time.time() - start
#
# start = time.time()
# spark.sql("SELECT * FROM polaris.ingestion.taxi_mor").collect()
# mor_read = time.time() - start
#
# print(f"\n{'Metric':<15} {'COW':>10} {'MOR':>10} {'Winner':>10}")
# print("-" * 50)
# print(f"{'Write':.<15} {cow_write:>9.3f}s {mor_write:>9.3f}s {'MOR' if mor_write < cow_write else 'COW':>10}")
# print(f"{'Read':.<15} {cow_read:>9.3f}s {mor_read:>9.3f}s {'COW' if cow_read < mor_read else 'MOR':>10}")

## Part 3: Row-Level Operations

### DELETE Operations

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_partitioned")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_partitioned
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT VendorID, tpep_pickup_datetime, passenger_count, trip_distance,
              fare_amount, tip_amount, total_amount
       FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned").collect()[0][0]
print(f"Partitioned table created with {count:,} trips")

In [ ]:
bad_fares = spark.sql("""
    SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned WHERE fare_amount <= 0
""").collect()[0][0]

start = time.time()
spark.sql("DELETE FROM polaris.ingestion.taxi_partitioned WHERE fare_amount <= 0")
delete_time = time.time() - start

remaining = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned").collect()[0][0]
print(f"Deleted {bad_fares:,} bad-fare trips in {delete_time:.3f} seconds")
print(f"Remaining: {remaining:,} trips")

### MERGE Operations (Upserts)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_corrections")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_corrections
    USING iceberg
    AS SELECT VendorID, tpep_pickup_datetime, passenger_count,
              trip_distance, fare_amount, tip_amount, total_amount
       FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
       LIMIT 1000
""")

print(f"Corrections table created with 1,000 trips")

In [ ]:
corrections = spark.sql("""
    SELECT VendorID, tpep_pickup_datetime, passenger_count,
           trip_distance, fare_amount * 1.05 as fare_amount,
           tip_amount, total_amount * 1.05 as total_amount
    FROM polaris.ingestion.taxi_corrections
    LIMIT 100
""")
corrections.createOrReplaceTempView("fare_corrections")

print("Prepared 100 fare corrections (5% increase)")

In [ ]:
spark.sql("""
    MERGE INTO polaris.ingestion.taxi_corrections t
    USING fare_corrections s
    ON t.tpep_pickup_datetime = s.tpep_pickup_datetime
       AND t.VendorID = s.VendorID
       AND t.trip_distance = s.trip_distance
    WHEN MATCHED THEN UPDATE SET
        t.fare_amount = s.fare_amount,
        t.total_amount = s.total_amount
    WHEN NOT MATCHED THEN INSERT *
""")

print("MERGE completed!")

In [ ]:
print("Corrections table after MERGE:")
spark.sql("""
    SELECT COUNT(*) as total_rows,
           ROUND(AVG(fare_amount), 2) as avg_fare,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM polaris.ingestion.taxi_corrections
""").show()

### Try It: Write Your Own MERGE

Create a small corrections dataset (e.g., adjust `tip_amount` for specific trips) and MERGE it into `taxi_corrections`. Try using both `WHEN MATCHED THEN UPDATE` and `WHEN NOT MATCHED THEN INSERT` clauses. Check the snapshot history to see the MERGE operation recorded.

In [ ]:
# Create your own corrections — adjust tip_amount by some factor
# my_corrections = spark.sql("""
#     SELECT VendorID, tpep_pickup_datetime, passenger_count,
#            trip_distance, fare_amount, tip_amount * 1.2 as tip_amount, total_amount
#     FROM polaris.ingestion.taxi_corrections
#     LIMIT 50
# """)
# my_corrections.createOrReplaceTempView("my_corrections")
#
# spark.sql("""
#     MERGE INTO polaris.ingestion.taxi_corrections t
#     USING my_corrections s
#     ON t.tpep_pickup_datetime = s.tpep_pickup_datetime
#        AND t.VendorID = s.VendorID
#        AND t.trip_distance = s.trip_distance
#     WHEN MATCHED THEN UPDATE SET t.tip_amount = s.tip_amount
#     WHEN NOT MATCHED THEN INSERT *
# """)
#
# spark.sql("""
#     SELECT committed_at, operation
#     FROM polaris.ingestion.taxi_corrections.snapshots
#     ORDER BY committed_at
# """).show(truncate=False)

## Part 4: Concurrency and Conflicts

Traditional databases use locks (pessimistic concurrency) to prevent conflicts — two transactions can update different rows in the same table without blocking each other. Iceberg uses **optimistic concurrency** instead: writers proceed independently and only check for conflicts at commit time. Conflicts are detected at the file level — if two writers modify files in the same partition, the second commit fails and must retry. Writes to different partitions do not conflict.

### Non-Conflicting Writes

Updates to different partitions succeed without conflict.

In [ ]:
print("Updating trips on two different days...")

start = time.time()
spark.sql("""
    UPDATE polaris.ingestion.taxi_partitioned
    SET fare_amount = fare_amount * 1.02
    WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
""")
t1 = time.time() - start

start = time.time()
spark.sql("""
    UPDATE polaris.ingestion.taxi_partitioned
    SET fare_amount = fare_amount * 1.03
    WHERE tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
""")
t2 = time.time() - start

print(f"Update June 1:  {t1:.3f}s")
print(f"Update June 15: {t2:.3f}s")

Both updates succeeded because they touched different partitions, so there was no conflict.

### Understanding Conflicts

Concurrent writes to the **same** partition will conflict:

```
Writer 1: UPDATE taxi SET fare = fare + 1 WHERE pickup_date = '2023-06-01'
Writer 2: UPDATE taxi SET fare = fare * 1.1 WHERE pickup_date = '2023-06-01'
```

**Result:** One writer gets a `CommitFailedException` and must retry with fresh data.

> **Note:** Triggering a real conflict requires concurrent Spark sessions, which is impractical in a single notebook. In production, you'd see a `CommitFailedException` when two jobs try to update the same partition.

**Best practice:** Include partition columns in `MERGE ON` clauses so Iceberg can isolate conflicts by partition.

### Try It: Test Partition Isolation

Update rows in two different partitions of `taxi_partitioned` and verify that each operation succeeds independently. Then try updating the same partition twice in sequence — how does the snapshot history reflect each write?

In [ ]:
# Pick two different days
# day_1 = "'2023-06-10'"
# day_2 = "'2023-06-20'"

# Update each partition
# spark.sql(f"""
#     UPDATE polaris.ingestion.taxi_partitioned
#     SET fare_amount = fare_amount * 1.01
#     WHERE CAST(tpep_pickup_datetime AS DATE) = {day_1}
# """)
# spark.sql(f"""
#     UPDATE polaris.ingestion.taxi_partitioned
#     SET fare_amount = fare_amount * 1.01
#     WHERE CAST(tpep_pickup_datetime AS DATE) = {day_2}
# """)

# Check snapshot history
# spark.sql("""
#     SELECT committed_at, operation, summary['changed-partition-count'] as partitions_touched
#     FROM polaris.ingestion.taxi_partitioned.snapshots
#     ORDER BY committed_at DESC LIMIT 5
# """).show(truncate=False)

## Cleanup

In [ ]:
# Optional: Drop tables
# for table in ['taxi_trips', 'taxi_cow', 'taxi_mor', 'taxi_partitioned', 'taxi_corrections']:
#     spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.{table}")
# print("Tables dropped!")